# AI 면접관 Agent
- 내 서류와 경험을 다시 정리하고 → 그 경험을 바탕으로 질문하고 → 꼬리질문으로 깊게 파고들고 → 답변을 평가하고 피드백해주는 Agent

## 1. 환경 설정 및 라이브러리 로딩

In [ ]:
# ============================================================
# 1. 공통 설정 및 모듈 경로
# ============================================================

import json
import sys
import uuid
from pathlib import Path
from typing import Any, Optional

from dotenv import load_dotenv
from langgraph.types import Command

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")


## 2. Interview State 설계

### 목적
- 각 Node의 입출력과 면접 진행 상태를 하나의 State로 관리
- 질문 → 답변 → 평가 → Router의 반복 흐름을 위한 정보 저장

### 입출력
- **입력** : 이력서/자기소개서, 채용공고
- **출력** : 최종 면접 피드백

### 처리
- 각 Node가 필요한 정보를 State에서 가져와 처리
- Node 처리 결과를 다시 State에 저장
- 답변 평가 결과에 따라 `FOLLOW_UP / NEXT / END` 흐름 관리

### State 관리 정보
- 입력 : 서류, 채용공고
- 분석 : 파싱 결과, 지원자/JD 분석
- 면접 준비 : RAG 결과, 면접 전략
- 인터뷰 : 현재 질문/답변, 질문·답변 기록
- 평가 : 답변 평가 결과
- 진행 상태 : `FOLLOW_UP / NEXT / END`
- 결과 : 최종 피드백

In [ ]:
# 전체 LangGraph 공유 상태
from src.state import InterviewState


## 3. 문서 파싱 설계

### 목적
- 서로 다른 구조의 서류와 채용공고에서 분석에 필요한 Text를 손실 없이 추출

### 입출력
- **입력** : 이력서/자기소개서 PDF, 채용공고 URL
- **출력** : 서류 Text, 채용공고 Text

### 처리
- PDF의 페이지/표/본문 Text 추출
- 채용공고 URL에서 실제 공고 본문 추출
- 불필요한 UI·메뉴·개인정보 등 제거
- 이후 분석 Node에서 활용할 수 있도록 Text 구조 정리

### 관리 정보
- `resume_file_path`
- `job_posting_url`
- `resume_text`
- `job_description_text`
- `parsing_error`

### 비고
- **PDF** : 페이지별 본문 Text를 추출하고 페이지 번호를 함께 저장한다. 표는 별도 구조로 추출하지 않으며, PDF의 Text Layer에 포함된 경우 일반 Text 형태로 추출된다. 이미지와 스캔 문서는 현재 추출하지 않는다.
- **사람인 URL** : 회사명, 공고명, 채용공고 상세 본문을 추출한다. 기본 상세 페이지 추출 실패 시 모바일 페이지를 사용한다.
- **원티드 URL** : 회사명, 포지션명, 회사·팀 소개, 주요 업무, 자격 요건, 우대 사항, 혜택, 기술 태그, 근무 지역 및 채용 전형 안내를 추출한다.
- 공통으로 메뉴·버튼·스크립트 등의 UI Text를 제거하고 이메일과 전화번호는 마스킹한다.

In [ ]:
# 문서 파싱 공통 기능
from src.services import normalize_text, redact_personal_info, resolve_project_path


In [ ]:
# 이력서 PDF 파싱
from src.services.document_parser import parse_resume_pdf


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# 사람인 채용공고 파싱
from src.services.job_posting_parser import extract_saramin_rec_idx, parse_saramin_job


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# 원티드 및 통합 채용공고 파싱
from src.services.job_posting_parser import (
    extract_wanted_job_id,
    parse_job_posting,
    parse_wanted_job,
)


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# 문서 파싱 LangGraph Node
from src.nodes.document_parsing import document_parsing_node


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 4. 파싱 결과 검증 설계

### 목적
- 추출된 서류와 채용공고 Text가 이후 LLM 분석에 사용할 수 있는 품질인지 확인

### 입출력
- **입력** : `resume_text`, `job_description_text`
- **출력** : 파싱 검증 결과 `PASS / FAIL`

### 처리
- Text가 비어 있거나 지나치게 짧은지 확인
- 문자 깨짐 등 비정상 Text 여부 확인
- PDF와 채용공고 각각의 파싱 성공 여부 확인
- 실패한 경우 재파싱 대상으로 전달

### 관리 정보
- `resume_parsing_status`
- `job_parsing_status`
- `resume_parsing_error`
- `job_parsing_error`

### 비고
- 이력서와 채용공고 Text가 비어 있지 않고 각각 최소 100자 이상인지 검증한다.
- NULL 문자(`\x00`)가 포함되어 있는지 확인한다.
- 문자 깨짐을 나타내는 대체 문자(`�`)가 전체 Text의 약 1%를 초과하는지 확인한다.
- 한글·영문·숫자 등 유효 문자의 비율이 전체 Text의 15% 이상인지 확인한다.
- 이력서와 채용공고가 모두 `PASS`인 경우에만 다음 분석 단계로 이동하며, 하나라도 실패하면 오류 내용을 저장하고 재파싱 대상으로 처리한다.

In [ ]:
# 파싱 결과 검증 및 분기
from src.nodes.parsing_validation import (
    parsing_validation_node,
    route_after_parsing_validation,
    validate_extracted_text,
)


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 5. 서류·채용공고 분석 / 구조화

### 목적
- 파싱된 이력서/자기소개서와 채용공고 내용을 LLM이 분석하여 면접 전략 수립에 사용할 수 있는 구조화된 정보로 변환

### 입출력
- **입력** : `resume_text`, `job_description_text`
- **출력** : `candidate_profile`, `jd_analysis`

### 처리
- 이력서/자기소개서에서 프로젝트, 경험, 기술, 역할, 성과 등 지원자 정보 추출
- 채용공고에서 주요 업무, 자격요건, 우대사항, 요구 기술 등 직무 정보 추출
- LLM + Structured Output을 활용해 정해진 Schema 형태로 구조화
- 면접 평가나 질문 방향은 결정하지 않고 문서에 존재하는 사실 중심으로 정리

### 비고
- **Schema 형식** : `candidate_profile`과 `jd_analysis`가 정의된 Pydantic Schema의 필드명과 자료형을 지키는지 검증한다.
- **지원자 정보** : 경력 요약, 경험, 프로젝트, 기술, 자격증이 올바른 항목으로 분리되어 있는지 확인한다. 프로젝트는 역할·수행 작업·사용 기술·성과가 구분되어야 한다.
- **채용공고 정보** : 회사명, 포지션, 주요 업무, 필수 자격요건, 우대사항, 요구 기술, 근무지 및 채용 절차가 올바른 항목으로 분리되어 있는지 확인한다.
- **근거 일치성** : 구조화된 내용이 원본 서류와 채용공고에 실제로 존재하며, 문서에 없는 경력·기술·성과를 생성하지 않았는지 확인한다.
- **누락·중복·분류 오류** : 원문에 명시된 핵심 정보가 빠지거나 중복되지 않았는지, 필수요건과 우대사항 등이 서로 잘못 분류되지 않았는지 확인한다.
- 두 분석 결과가 모두 Schema와 원문 근거 검증을 통과한 경우에만 다음 면접 전략 수립 단계로 전달한다.

In [ ]:
# 지원자·채용공고 분석 Schema
from src.schemas.analysis import (
    CandidateExperience,
    CandidateProfile,
    CandidateProject,
    JobDescriptionAnalysis,
)


In [ ]:
# 지원자·채용공고 분석 Chain 및 Node
from src.chains.analysis import (
    candidate_analysis_prompt,
    get_candidate_analysis_chain,
    get_jd_analysis_chain,
    jd_analysis_prompt,
)
from src.nodes.candidate_jd_analysis import candidate_jd_analysis_node


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 6. 면접 전략 수립

### 목적

- 지원자 경험과 채용공고 요구사항을 비교해 면접에서 검증할 핵심 역량을 선정
- 각 역량의 연결 상태, 우선순위, 선정 근거, 검증 항목 및 질문 방향을 구성
- 실제 면접 질문을 생성하기 전 면접 전체의 진행 전략 수립

### 입출력

- **입력**: `candidate_profile`, `jd_analysis`, `retrieved_knowledge`
- **출력**: `interview_strategy`, `target_competencies`

### 처리

- 지원자 경험과 채용공고의 주요 업무·자격요건을 연결
- 서로 다른 평가 목적을 가진 핵심 역량 3~5개 선정
- 각 역량의 연결 상태를 `MATCH`, `PARTIAL`, `UNVERIFIED`로 분류
- 연결이 부족하거나 확인되지 않은 항목을 `gap_to_verify`에 정리
- 역량별 우선순위와 선정 근거 설정
- 지원자 정보와 채용공고에서 확인 가능한 근거 연결
- 면접에서 확인할 세부 항목과 질문 방향 정의
- 실제 질문 문장은 생성하지 않고 면접 전략만 구성
- 검증된 역량을 우선순위 순으로 정렬해 `target_competencies` 생성
- `target_competencies`를 기준으로 화면 표시용 `interview_flow` 생성

> 현재 RAG는 연결하지 않았으며 `retrieved_knowledge=[]`을 사용한다.

### 연결 상태 기준

- **MATCH**: 지원자 경험이 직무 요구사항과 직접 연결되는 경우
- **PARTIAL**: 관련 경험은 있지만 직무 요구사항을 충분히 확인하기 어려운 경우
- **UNVERIFIED**: 지원자 정보에서 관련 경험을 확인할 수 없는 경우
- `PARTIAL`과 `UNVERIFIED`는 추가 확인할 내용을 `gap_to_verify`에 작성한다.
- `UNVERIFIED`인 경우 `candidate_evidence`는 빈 목록으로 유지한다.

### 검증 기준

- **Schema 형식**: `interview_strategy`가 `InterviewStrategyPlan`과 `CompetencyStrategy`의 필드 및 자료형을 준수하는지 확인한다.
- **핵심 역량**: `competencies`가 서로 다른 평가 목적을 가진 3~5개 역량으로 생성되었는지 확인한다.
- **중복 방지**: 동일한 역량명이 중복되지 않았는지 확인하며, 의미가 유사한 역량은 프롬프트에서 중복 생성을 방지한다.
- **우선순위**: `priority`가 1부터 역량 수까지 중복 없이 연속되는지 확인한다.
- **연결 상태**: `alignment_status`, `candidate_evidence`, `gap_to_verify`의 조합이 연결 상태 기준과 일치하는지 확인한다.
- **지원자 근거**: `candidate_evidence`가 `candidate_profile`의 내용으로 뒷받침되는지 확인한다. 근거가 없으면 빈 목록을 허용한다.
- **채용공고 근거**: `jd_evidence`가 `jd_analysis`의 내용으로 뒷받침되며 역량별로 최소 1개 존재하는지 확인한다.
- **근거 일치 방식**: 입력과 정확히 일치하는 문구 또는 입력의 여러 사실을 결합한 근거를 허용하되, 새로운 경험·기술·성과·수치는 허용하지 않는다.
- **검증 항목**: `verification_points`가 역량별로 1~3개 생성되는지 확인한다.
- **질문 방향**: `verification_points`와 `question_direction`이 실제 질문문이 아닌 명사형 확인 항목이며 물음표를 포함하지 않는지 확인한다.
- **진행 순서**: `competencies`, `target_competencies`, `interview_flow`가 동일한 우선순위 순서를 따르는지 확인한다.
- **RAG 미연결 처리**: `retrieved_knowledge`가 비어 있으면 `rag_applied=False`, `rag_basis=[]`로 유지한다.

### 실패 처리

- Structured Output 결과가 검증 조건을 통과하면 다음 질문 생성 단계로 전달한다.
- 검증에 실패하면 오류 내용을 보정 지시로 전달해 같은 Node 안에서 최대 1회 다시 생성한다.
- 재생성 결과도 검증에 실패하면 `ValueError`를 발생시키고 다음 단계로 전달하지 않는다.

In [ ]:
# 면접 전략 Schema
from src.schemas.strategy import CompetencyStrategy, InterviewStrategyPlan


In [ ]:
# 면접 전략 Chain 및 Node
from src.chains.strategy import (
    get_interview_strategy_chain,
    interview_strategy_prompt,
)
from src.nodes.interview_strategy import interview_strategy_node


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 7. 질문 생성

### 목적
- 면접 전략의 연결 상태와 현재 진행 상황을 반영해 지원자에게 적합한 개인화 질문 생성
- 실제 확인된 근거와 아직 확인할 공백을 구분해 질문 목적 설정

### 입출력
- **입력** : `interview_strategy`, `target_competencies`, `candidate_profile`, `jd_analysis`, `route`, `interview_history`, `current_evaluation`
- **출력** : `current_question`, `current_competency`, `question_type`

### 처리
- 면접 전략의 우선순위에 따라 현재 평가할 역량 선택
- `MATCH`는 확인된 경험의 역할·행동·기술적 판단·성과를 깊게 확인
- `PARTIAL`은 관련 경험을 바탕으로 `gap_to_verify`의 부족한 부분 한 가지를 확인
- `UNVERIFIED`는 관련 경험을 단정하지 않고 경험 유무를 중립적으로 확인
- `INITIAL`과 `NEXT`는 현재 역량의 연결 상태에 맞는 새로운 질문 생성
- `FOLLOW_UP`은 연결 상태보다 직전 답변의 `missing_points` 한 가지를 우선 확인
- 이미 질문한 내용을 반복하지 않도록 `interview_history` 참고
- 한 번에 하나의 질문만 생성
- 중복 또는 복수 질문이 생성되면 같은 Node 안에서 최대 1회 보정해 다시 생성

### 비고
- **Schema 형식** : `current_question`, `current_competency`, `question_type`이 모두 생성되고 정의된 자료형을 지키는지 확인한다. `question_type`은 `INITIAL`, `NEXT`, `FOLLOW_UP` 중 하나여야 한다.
- **질문 유형과 역량 선택** : 최초 질문은 `INITIAL`이면서 우선순위가 가장 높은 역량을 선택하고, `NEXT`는 아직 질문하지 않은 다음 역량을 선택하는지 확인한다. `FOLLOW_UP`은 이전 질문과 동일한 역량을 유지해야 한다.
- **연결 상태별 질문** : `MATCH`, `PARTIAL`, `UNVERIFIED`에 따라 경험 심층 확인, 공백 확인, 경험 유무 확인으로 질문 목적이 구분되는지 확인한다.
- **개인화 및 근거 일치성** : 질문이 `candidate_evidence`, `jd_evidence`와 사용자 답변에 있는 사실을 근거로 하며, 문서에 없는 경험·기술·역할·성과·수치를 단정하지 않았는지 확인한다.
- **꼬리질문 연결성** : `FOLLOW_UP` 질문이 이전 답변과 `current_evaluation`의 부족한 항목 중 한 가지를 구체적으로 확인하며, 이전 답변과 무관한 주제로 바뀌지 않았는지 확인한다.
- **단일 질문과 문장 품질** : `current_question`이 비어 있지 않고 물음표로 끝나는 자연스러운 한국어 의문문인지, 한 문장에 여러 질문을 결합하거나 답변 예시·평가 결과를 노출하지 않았는지 확인한다.
- **중복 방지** : 생성된 질문이 `interview_history`의 기존 질문과 같거나 동일한 내용을 반복하지 않는지 확인한다.
- 모든 조건을 통과한 질문만 답변 수집 단계로 전달하고, 실패한 경우 현재 역량과 질문 유형을 유지한 상태에서 최대 1회 보정해 다시 생성한다.

In [ ]:
# 질문 생성 Schema
from src.schemas.interview import GeneratedInterviewQuestion


In [ ]:
# 질문 생성 Chain 및 Node
from src.chains.question import get_question_generation_chain, question_generation_prompt
from src.nodes.question_generation import question_generation_node, select_question_context


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 8. 사용자 답변

### 목적
- 생성된 질문을 사용자에게 전달하고, 입력받은 답변을 다음 답변 평가 단계에서 사용할 수 있도록 State에 저장

### 입출력
- **입력** : `current_question`, `current_competency`, `question_type`, `interview_history`, `question_count`, `followup_count`
- **출력** : `current_answer`, `interview_history`, `question_count`, `followup_count`

### 처리
- 현재 질문·평가 역량·질문 유형을 사용자에게 전달하고 `interrupt()`로 답변 입력을 대기
- 같은 `thread_id`에서 `Command(resume=...)`로 전달된 답변을 정리해 `current_answer`에 저장
- 현재 질문과 답변 정보를 `interview_history`에 추가
- 답변 완료 수와 꼬리질문 답변 수를 갱신한 뒤 답변 평가 단계로 전달

### 비고
- **LLM 미사용** : 사용자 입력을 저장하는 단계이므로 별도의 Prompt나 모델을 사용하지 않는다.
- **상태 저장** : `interrupt()`로 중단된 그래프를 재개하려면 Checkpointer와 동일한 `thread_id`가 필요하다.
- **상태 저장소** : `DATABASE_URL`이 있으면 PostgreSQL에 영구 저장하고, 없으면 개발·테스트용 `InMemorySaver`를 사용한다.
- **필수 상태** : `current_question`, `current_competency`, `question_type`이 없거나 질문 유형이 올바르지 않으면 오류를 반환한다.
- **빈 답변 처리** : 빈 답변은 State에 저장하지 않고 오류를 반환하며, 실제 입력 화면에서 다시 입력받는다.
- **기록과 카운터** : 답변 저장 시 면접 기록과 `question_count`를 갱신하고, `FOLLOW_UP` 답변일 때만 `followup_count`를 증가시킨다.

In [ ]:
# 사용자 답변 수집 Node
from src.nodes.user_answer import normalize_user_answer, user_answer_node


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 9. 답변 평가

### 목적
- 사용자의 답변을 공통 기준으로 평가하고, 강점과 부족한 부분을 구조화하여 다음 면접 진행 판단에 활용

### 입출력
- **입력** : `current_question`, `current_answer`, `current_competency`, `question_type`, `interview_strategy`, `evaluation_history`
- **출력** : `current_evaluation`, `evaluation_history`

### 처리
- 현재 질문과 역량별 검증 항목을 기준으로 다음 6개 항목을 각각 1~5점으로 평가

| 평가 항목 | 기준 | 배점 |
|---|---|---:|
| 질문 관련성 | 질문 의도와 평가 역량에 맞게 답변했는지 | 20 |
| 답변 구체성 | 상황, 사례, 기술, 수치가 구체적인지 | 15 |
| 역할 명확성 | 지원자가 직접 맡은 역할과 기여가 명확한지 | 15 |
| 행동·판단 | 수행한 행동과 판단 근거가 명확한지 | 20 |
| 결과·학습 | 성과, 변화, 실패 결과 또는 학습이 명확한지 | 20 |
| 논리적 전달력 | 답변 흐름이 자연스럽고 이해하기 쉬운지 | 10 |

- `INITIAL`, `NEXT`는 현재 답변을 중심으로 평가
- `FOLLOW_UP`은 동일 역량의 이전 답변과 현재 답변을 함께 평가
- 세부 점수에 배점을 적용해 100점 기준 `overall_score` 계산
- 답변에서 확인된 강점, 보완점, 누락 항목, 핵심 근거를 구조화
- `FOLLOW_UP / NEXT / END`는 결정하지 않고 평가 결과만 생성

### 비고
- **평가 근거** : 현재 답변과 동일 역량의 이전 답변에 실제로 포함된 내용만 사용한다.
- **연결 상태 분리** : `MATCH`, `PARTIAL`, `UNVERIFIED`는 질문 방향에만 사용하고 답변 점수에는 직접 반영하지 않는다.
- **빈 목록 허용** : 확인할 수 없는 강점, 보완점, 누락 항목, 답변 근거는 억지로 만들지 않고 빈 목록을 허용한다.
- **점수 계산** : LLM은 1~5점의 세부 점수만 생성하고, `overall_score`는 코드에서 배점에 따라 계산한다.
- **평가 이력** : 질문, 답변, 역량, 질문 유형과 평가 결과를 `evaluation_history`에 추가한다.
- **역할 분리** : 다음 진행 경로는 생성하지 않고 다음 인터뷰 진행 검토 Node에 맡긴다.
- **검증 범위** : 현재는 필수 상태와 Structured Output 형식만 확인하며, 자동 재생성과 전체 품질 검증은 추후 통합 단계에서 수행한다.

In [ ]:
# 답변 평가 Schema
from src.schemas.interview import AnswerEvaluationResult, AnswerEvaluationScores


In [ ]:
# 답변 평가 Chain 및 Node
from src.chains.evaluation import answer_evaluation_prompt, get_answer_evaluation_chain
from src.nodes.answer_evaluation import answer_evaluation_node, calculate_overall_score


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 10. 인터뷰 진행 검토

### 목적
- 현재 답변의 충분성과 전체 면접 진행 상태를 확인하여 다음 질문 또는 종료 여부를 결정

### 입출력
- **입력** : `current_evaluation`, `current_competency`, `target_competencies`, `interview_history`, `question_count`, `followup_count`
- **출력** : `route`, `end_reason`

### 처리
- `overall_score`가 70점 이상이고 `missing_points`가 없으면 현재 답변을 충분한 것으로 판단
- 점수가 70점 미만이거나 누락 항목이 있으면 같은 역량의 `FOLLOW_UP` 가능 여부 확인
- 답변이 충분하거나 꼬리질문 제한에 도달하면 아직 확인하지 않은 다음 역량으로 `NEXT`
- 모든 핵심 역량을 확인했거나 전체 질문이 최대 10개에 도달하면 `END`
- 남은 질문 수가 부족하면 꼬리질문보다 아직 확인하지 않은 핵심 역량을 우선

### 비고
- **기본 질문 수** : 핵심 역량별로 한 번씩 질문하므로 기본 질문은 3~5개이며, 답변 상태에 따라 꼬리질문이 추가된다.
- **전체 질문 제한** : 최대 10개는 반드시 채워야 하는 목표가 아니라 초과를 막는 상한선이며, 모든 역량 확인이 끝나면 10개 전에도 종료한다.
- **꼬리질문 제한** : 동일 역량에서는 최대 2개까지만 허용하고, 제한에 도달하면 다음 역량으로 이동하거나 종료한다.
- **답변 충분 기준** : 가중 종합점수 70점 이상과 누락 항목 없음 두 조건을 모두 충족해야 한다.
- **남은 역량 우선** : 전체 질문 제한 안에서 모든 핵심 역량을 최소 한 번씩 확인할 수 있도록 질문 자리를 남긴다.
- **카운터 확인** : `question_count`는 전체 면접 기록 수보다, `followup_count`는 기록된 꼬리질문 수보다 작지 않은지 확인한다.
- **LLM 미사용** : 정해진 조건문만으로 `FOLLOW_UP`, `NEXT`, `END`를 결정한다.
- **검증 범위** : 실제 면접에서는 상태값과 횟수만 확인하며, 전체 Router 품질 검증은 별도 테스트에서 수행한다.

In [ ]:
# 인터뷰 진행 검토 규칙 및 Node
from src.config import (
    MAX_FOLLOWUPS_PER_COMPETENCY,
    MAX_INTERVIEW_QUESTIONS,
    MIN_SUFFICIENT_SCORE,
)
from src.nodes.interview_review import interview_review_node
from src.schemas.interview import InterviewRouteDecision


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 11. 인터뷰 피드백 보고서

### 목적
- 각 질문에 대한 개별 피드백과 전체 면접에 대한 종합 피드백을 함께 제공

### 입출력
- **입력** : `interview_history`, `evaluation_history`, `interview_strategy`, `target_competencies`
- **출력** : `final_feedback`

### 처리
- `evaluation_history`를 사용해 질문별 점수, 강점, 보완점, 누락 항목, 개선 초점을 코드로 구성
- 꼬리질문이 있는 역량은 가장 마지막 누적 평가점수를 해당 역량의 최종 점수로 사용
- 모든 역량의 최종 점수를 동일하게 평균하여 전체 점수 계산
- 역량별 피드백, 전체 강점, 우선 개선점, 반복 패턴, 실행 계획을 종합
- 질문별 피드백은 기존 평가 결과를 재사용하고, LLM은 전체 피드백 생성에만 한 번 사용
- 실제 질문·답변·평가에 없는 경험, 수치, 성과 또는 판단은 생성하지 않음

### 비고
- **질문별 피드백** : 질문, 답변, 역량, 질문 유형, 점수, 요약, 강점, 보완점, 누락 항목, 개선 초점을 포함한다.
- **역량별 최종 점수** : 동일 역량의 평가가 여러 개이면 가장 마지막 평가점수를 사용하여 꼬리질문 횟수에 따른 점수 편향을 방지한다.
- **전체 피드백** : 전체 점수, 역량별 점수, 종합 요약, 주요 강점, 우선 개선점, 반복 패턴, 실행 계획과 면접 통계를 포함한다.
- **실행 계획** : 확인된 개선점이 있을 때만 1~3개의 구체적인 연습 행동과 목적을 작성한다.
- **빈 목록 허용** : 확인할 수 없는 강점, 개선점, 근거, 반복 패턴은 억지로 만들지 않고 빈 목록을 허용한다.
- **판단 제한** : 답변 개선 목적에 집중하고 합격·불합격, 성격, 태도 등 확인할 수 없는 판단은 생성하지 않는다.
- **표시 시점** : 실제 면접 흐름을 방해하지 않도록 질문별 피드백과 전체 피드백은 면접 종료 후 함께 제공한다.
- **검증 범위** : 실제 실행에서는 필수 이력과 역량 목록만 확인하며, 전체 보고서 품질 검증은 추후 별도 테스트에서 수행한다.

In [ ]:
# 인터뷰 피드백 보고서 Schema
from src.schemas.interview import (
    CompetencyFeedback,
    FeedbackActionItem,
    FinalInterviewFeedbackReport,
    QuestionFeedback,
)


In [ ]:
# 인터뷰 피드백 Chain 및 Node
from src.chains.feedback import feedback_report_prompt, get_feedback_report_chain
from src.nodes.final_feedback import (
    build_question_feedback,
    calculate_feedback_scores,
    final_feedback_node,
)


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


## 12. LangGraph 연결 설계

### 전체 흐름

`START → 문서 파싱 → 파싱 결과 검증 → 서류·채용공고 분석 → 면접 전략 수립`

`→ 질문 생성 → 사용자 답변 → 답변 평가 → 인터뷰 진행 검토`

`→ 질문 생성 반복 또는 피드백 보고서 → END`


### 조건 분기

#### 파싱 결과 검증
- `PASS` → 서류·채용공고 분석으로 이동
- `RETRY` → 문서 파싱 재시도
- 최대 2회 모두 실패하면 오류를 반환하고 종료

#### 인터뷰 진행 검토
- `FOLLOW_UP` → 현재 역량의 부족한 답변을 보완하는 꼬리질문 생성
- `NEXT` → 우선순위상 다음 역량의 질문 생성
- `END` → 질문별 피드백과 전체 피드백 보고서 생성


### 반복 흐름

- 한 번에 질문 하나를 생성하고 답변과 평가 결과를 이력에 누적
- 평가 점수가 70점 미만이거나 누락 항목이 있으면 현재 역량의 꼬리질문 검토
- 답변이 충분하거나 역량별 꼬리질문이 2개에 도달하면 다음 역량으로 이동
- 전체 질문은 최대 10개이며, 남은 역량을 최소 한 번씩 확인할 질문 수를 우선 확보


### 사용자 입력 대기

- 질문 생성 후 `interrupt`로 Graph를 중단하고 사용자 답변을 대기
- 동일한 `thread_id`와 `Command(resume={"answer": ...})`로 중단 지점부터 재개
- `DATABASE_URL`이 있으면 PostgreSQL Checkpointer를 사용하고, 없으면 개발·테스트용 `InMemorySaver` 사용


### 종료 조건

- 모든 핵심 역량을 최소 한 번씩 확인하고 현재 역량의 추가 질문이 필요하지 않은 경우
- 모든 역량 확인 후 현재 역량의 꼬리질문 제한에 도달한 경우
- 전체 질문이 최대 10개에 도달한 경우

종료 후 질문별 피드백과 전체·역량별 피드백을 하나의 `final_feedback`으로 반환한다.

In [ ]:
# 전체 LangGraph 연결 및 개발용 Checkpointer
from src.graph.interview_graph import (
    build_interview_graph,
    interview_checkpointer,
    interview_graph,
    route_after_interview_review,
    route_after_pipeline_parsing_validation,
)


In [ ]:
# 실행 테스트는 tests/ 디렉터리에서 관리합니다.


In [ ]:
# ============================================================
# 12-3. 전체 LangGraph 연결 구조 시각화
# ============================================================

# 외부 렌더링 호출 없이 Mermaid 구조만 확인합니다.
graph_view = interview_graph.get_graph()
print(graph_view.draw_mermaid())

In [ ]:
# ============================================================
# 12-4. 실제 사용자 답변 기반 인터뷰 실행 예시
# ============================================================


def print_final_feedback(final_feedback: dict[str, Any]) -> None:
    """질문별 피드백과 전체 피드백을 구분해 출력한다."""
    question_feedback = final_feedback.get("question_feedback", [])
    if question_feedback:
        print("\n=== 질문별 피드백 ===")
        for item in question_feedback:
            print(
                f"[{item['question_number']}] "
                f"{item['competency']} · {item['overall_score']}점"
            )
            if item.get("summary"):
                print(f"- 평가: {item['summary']}")
            if item.get("improvement_focus"):
                print(f"- 우선 보완: {item['improvement_focus']}")

    print("\n=== 전체 피드백 ===")
    if final_feedback.get("overall_score") is not None:
        print(f"전체 점수: {final_feedback['overall_score']}점")
    if final_feedback.get("overall_summary"):
        print(final_feedback["overall_summary"])


def run_interview_interactively(
    resume_file_path: str,
    job_posting_url: str,
    thread_id: Optional[str] = None,
) -> InterviewState:
    """질문을 출력하고 사용자 답변을 받아 전체 인터뷰를 실행한다."""
    if not resume_file_path or not job_posting_url:
        raise ValueError("이력서 PDF 경로와 채용공고 URL이 필요합니다.")

    active_thread_id = thread_id or f"interview-{uuid.uuid4()}"
    config = {
        "configurable": {"thread_id": active_thread_id},
        "recursion_limit": 200,
    }
    result = interview_graph.invoke(
        {
            "resume_file_path": resume_file_path,
            "job_posting_url": job_posting_url,
            "retrieved_knowledge": [],
        },
        config=config,
    )

    while "__interrupt__" in result:
        interrupt_info = result["__interrupt__"][0].value
        print("\n" + "=" * 60)
        print(f"질문 유형: {interrupt_info.get('question_type', '')}")
        print(f"평가 역량: {interrupt_info.get('competency', '')}")
        print(f"면접관: {interrupt_info.get('question', '')}")

        answer = input("지원자 답변: ").strip()
        while not answer:
            answer = input("답변이 비어 있습니다. 다시 입력해 주세요: ").strip()
        result = interview_graph.invoke(
            Command(resume={"answer": answer}),
            config=config,
        )

    print(f"\n인터뷰 종료: {result.get('end_reason') or '완료'}")
    if result.get("final_feedback"):
        print_final_feedback(result["final_feedback"])
    return result


# 실제 면접을 시작할 때만 아래 호출의 주석을 해제합니다.
# final_interview_result = run_interview_interactively(
#     resume_file_path="data/incrut_sample.pdf",
#     job_posting_url="https://www.saramin.co.kr/zf_user/jobs/relay/view?rec_idx=54940771",
# )
